In [1]:
import os
import sys
import time

# Add the project root to Python path so we can import mu_bench modules
project_root = os.path.abspath('..')
sys.path.append(project_root)

# Now import the required modules
from kubernetes import client, config
from mu_bench.deployers.manager import K8sDeploymentManager

APP_DIR = "../APPS"

In [7]:
def wait_for_pods_ready(namespace, timeout=300):
    """Wait for all pods in namespace to be ready"""
    config.load_kube_config()
    k8s_core_api = client.CoreV1Api()
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            pods = k8s_core_api.list_namespaced_pod(namespace=namespace)
            if not pods.items:
                time.sleep(5)
                continue
                
            ready_count = 0
            for pod in pods.items:
                if pod.status.conditions:
                    for condition in pod.status.conditions:
                        if condition.type == 'Ready' and condition.status == 'True':
                            ready_count += 1
                            break
            
            if ready_count == len(pods.items):
                print(f"✅ All {ready_count} pods ready in {namespace}")
                return True
                
            print(f"⏳ Waiting... {ready_count}/{len(pods.items)} pods ready")
            time.sleep(5)
            
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(5)
    
    print(f"❌ Timeout waiting for pods in {namespace}")
    return False

def deploy_app(app_name):
    """Deploy a single app"""
    app_dir = f"{APP_DIR}/{app_name}"
    
    print(f"🚀 Deploying {app_name}...")
    mgr = K8sDeploymentManager.from_deployment_dir(app_dir)
    mgr.deploy()
    
    print(f"⏳ Waiting for pods in {mgr.namespace}...")
    wait_for_pods_ready(mgr.namespace)

def undeploy_app(app_name):
    """Undeploy a single app"""
    app_dir = f"{APP_DIR}/{app_name}"
    
    print(f"🗑️ Undeploying {app_name}...")
    mgr = K8sDeploymentManager.from_deployment_dir(app_dir)
    mgr.undeploy()

In [ ]:
undeploy_app("app-edge")
deploy_app("app-edge")


🗑️ Undeploying app-edge...
[INFO] Undeploying resources from '../APPS/app-edge/yamls'...
[INFO] Undeployment completed successfully.
🚀 Deploying app-edge...
[INFO] Ensuring namespace 'app-edge' exists...
[INFO] Labeled namespace 'app-edge' with istio-injection=enabled.
[INFO] Removing existing directory ../APPS/app-edge/yamls before YAML generation...
[INFO] Successfully removed ../APPS/app-edge/yamls.
[INFO] Ensured YAML directory ../APPS/app-edge/yamls exists.
[INFO] Generating YAMLs in '../APPS/app-edge/yamls'...
[INFO] Deploying resources from '../APPS/app-edge/yamls'...
[INFO] Deployment completed successfully.
⏳ Waiting for pods in app-edge...
⏳ Waiting... 6/10 pods ready
⏳ Waiting... 6/7 pods ready
✅ All 6 pods ready in app-edge


In [11]:
undeploy_app("app-edge-2")
deploy_app("app-edge-2")

🗑️ Undeploying app-edge-2...
[INFO] Undeploying resources from '../APPS/app-edge-2/yamls'...
[INFO] Undeployment completed successfully.
🚀 Deploying app-edge-2...
[INFO] Ensuring namespace 'app-edge-2' exists...
[INFO] Labeled namespace 'app-edge-2' with istio-injection=enabled.
[INFO] Removing existing directory ../APPS/app-edge-2/yamls before YAML generation...
[INFO] Successfully removed ../APPS/app-edge-2/yamls.
[INFO] Ensured YAML directory ../APPS/app-edge-2/yamls exists.
[INFO] Generating YAMLs in '../APPS/app-edge-2/yamls'...
[INFO] Deploying resources from '../APPS/app-edge-2/yamls'...
[INFO] Deployment completed successfully.
⏳ Waiting for pods in app-edge-2...
⏳ Waiting... 2/8 pods ready
⏳ Waiting... 10/11 pods ready
✅ All 11 pods ready in app-edge-2
